# Training the model using CNN

In the previous stage we succesfully inspected the data, checked its quality, and defined the data splits. In the following we will:
* cut every file into 2 second windows to feed into the model
* attach labels to every window (0 or 1)
* Stack them into an array CNN can consume (possibly normalize)
* Define the CNN architecture
* Write the training loop
* Track training vs. validation accuracy/loss per epoch 

In [1]:
import importlib
import numpy as np
import pandas as pd
from helpers import load, window
from constants import FILE_S, WIN_S
import dataset
import torch
from torch.utils.data import TensorDataset, DataLoader

dataset = importlib.reload(dataset)
files = dataset.files
splits = dataset.splits


## Cut every file into windows.
Each ~70 second file gives about 35 non-overlapping 2-second windows.
Windows can have different row counts because packet rate slightly differs between files, however CNN needs a fixed one. Here we pick the window size to be the smallest window length we can find.

In [2]:
def make_windows(files, label, T): #T is the # of rows every window must have
    X, y = [], [] #X will collect the inputs, y will collect the correct labels
    for f in files:
        df = load(f) #reads on csv file into a table and drops the two edge carriers (helper function)
        for start in range(0, FILE_S, WIN_S):
            w = window(df, start) #cuts out the 2-sec window starting from start
            if len(w) >= T: #safety check
                w = w[:T]
                X.append(w - np.median(w, axis=0)) #computes median for each subcarrier (appending the centered data around 0 to X)
                y.append(label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32) #turns the arrays into numpy arrays, float32 is the type PyTorch expects (also less memory)

'''Finding T - the smallest window length'''
all_files =files["empty"] + files["person"] #one big list
T = min(len(window(load(f), 10)) for f in all_files) #take a 2sec window starting at 10sec from each file by first loading it, then compare the length of that window (some packet rates are faster than other so more rows), then choose min 


## Next step is to build the splits

In [11]:
def build(part):
    Xe, ye = make_windows(splits["empty"][part],  0, T)
    Xp, yp = make_windows(splits["person"][part], 1, T)
    return np.concatenate([Xe, Xp]), np.concatenate([ye, yp])

X_train, y_train = build("train")
X_val,   y_val   = build("val")
X_test,  y_test  = build("test")

for name, X, y in (("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)):
    print(name, X.shape, y.shape, "empty:", int((y == 0).sum()), "person:", int((y == 1).sum()))
pd.DataFrame(X_train[0][:5,:5])

train (1995, 81, 47) (1995,) empty: 210 person: 1785
val (665, 81, 47) (665,) empty: 70 person: 595
test (665, 81, 47) (665,) empty: 70 person: 595


,0,1,2,3,4
0,2.432702,6.327237,-1.950626,1.491439,-5.763897
1,5.234189,4.724953,2.869820,0.488853,0.587975
2,6.750565,14.484240,7.391074,8.373919,3.815616
3,6.743281,5.205689,9.743284,10.189011,11.285751
4,12.209437,8.930522,11.832970,9.308620,11.742233


In [4]:
# Scale the data using stats only from training set to avoid data leakage
mean = X_train.mean()
std = X_train.std()
print("mean/std from train:", mean, std)

X_train_s = (X_train - mean) / std
X_val_s   = (X_val   - mean) / std
X_test_s  = (X_test  - mean) / std

print("train mean/std after scaling:", X_train_s.mean(), X_train_s.std())
print("The first window limited to the first 5 packets and 5 subcarriers:")
pd.DataFrame(X_train_s[0][:5, :5]) 

mean/std from train: 19.616455 155.3665
train mean/std after scaling: 0.0 1.0
The first window limited to the first 5 packets and 5 subcarriers:


,0,1,2,3,4
0,-0.110601,-0.085535,-0.138814,-0.116660,-0.163358
1,-0.092570,-0.095848,-0.107788,-0.123113,-0.122475
2,-0.082810,-0.033033,-0.078687,-0.072361,-0.101700
3,-0.082857,-0.092753,-0.063548,-0.060679,-0.053620
4,-0.047674,-0.068779,-0.050098,-0.066345,-0.050682


## Add the channel dimension
Reshapes the data from (N, T, 47) to (N, 1, T, 47), adding the "channel" dimension that PyTorch's CNN layers expect

In [ ]:
X_train_s = X_train_s[:, None, :, :] #makes shape go from (N,T,47) into (N,1,T,47) None means to insert a new dimension of size 1
X_val_s   = X_val_s[:, None, :, :]
X_test_s  = X_test_s[:, None, :, :]
print("The shape of X_train_s: ", X_train_s.shape)   # should be (N, 1, T, 47), N is the number of windows, 1 is the channel dimension this case just amplitude

The shape of X_train_s:  (1995, 1, 81, 47)


,0,1,2,3,4
0,-0.110601,-0.085535,-0.138814,-0.116660,-0.163358
1,-0.092570,-0.095848,-0.107788,-0.123113,-0.122475
2,-0.082810,-0.033033,-0.078687,-0.072361,-0.101700
3,-0.082857,-0.092753,-0.063548,-0.060679,-0.053620
4,-0.047674,-0.068779,-0.050098,-0.066345,-0.050682


In [6]:
#Converts the NumPy arrays into PyTorch tensors which the data format PyTorch's models and training tools operate on. 
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train,   dtype=torch.float32)

X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
y_val_t = torch.tensor(y_val,   dtype=torch.float32)

X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test,   dtype=torch.float32)

print(X_train_t.shape, X_train_t.dtype)

torch.Size([1995, 1, 81, 47]) torch.float32


# PyTorch's DataLoaders
Wraps the tensors into batches for training: groups windows into batches of 32, shuffles the training data each epoch, and hands out one batch at a time during the training loop.

In [7]:
#1 we pair up our X and y
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t,   y_val_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

#2 wrap each dataset in a DataLoader
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

for X_batch, y_batch in train_loader:
    print(X_batch.shape, y_batch.shape)
    break #look only at first batch for now

torch.Size([32, 1, 81, 47]) torch.Size([32])


## Defining the CNN

## Loss function and optimizer

## The training loop

## Plotting the curves

## Pick the Best Epoch

## Evaluate once on test

## Report Metricc